In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- hybrid_score_concat ---
FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_BEFORE = [
    SimpleNamespace(to_frame=lambda: pd.DataFrame({"document_id":[0],"score":[0.9]})),
    SimpleNamespace(to_frame=lambda: pd.DataFrame({"document_id":[1],"score":[0.7]})),
]
FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_GEN = [
    SimpleNamespace(to_frame=lambda: pl.DataFrame({"document_id":[0],"score":[0.9]})),
    SimpleNamespace(to_frame=lambda: pl.DataFrame({"document_id":[1],"score":[0.7]})),
]

# --- hybrid_score_frame ---

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_hybrid_score_concat(hybrid_scores):
    return pd.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores], ignore_index=True)
    return None

def before_hybrid_score_frame():
    return pd.DataFrame(
        {"document_id": [0], "score": [0.9]},
        index=[0],
    )
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_hybrid_score_concat(hybrid_scores):

    return pl.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores])

def gen_hybrid_score_frame():

    def to_frame(language_model_name, citation_to_language_candidates, citation_to_language, language_to_citation_candidates, language_to_citation):
        return pl.DataFrame(
            [
                {
                    "Language Model": language_model_name,
                    "Citation -> Language Candidates": round(
                        citation_to_language_candidates, ndigits=3
                    ),
                    "Citation -> Language Final": round(citation_to_language, ndigits=3),
                    "Language -> Citation Candidates": round(
                        language_to_citation_candidates, ndigits=3
                    ),
                    "Language -> Citation Final": round(language_to_citation, ndigits=3),
                }
            ]
        )
    return None

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: hybrid_score_concat ===

# L1 smoke – generated
try:
    _r = gen_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_GEN)
    print("✅ L1 smoke gen_hybrid_score_concat: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_hybrid_score_concat: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_BEFORE)
    print("✅ L1 smoke before_hybrid_score_concat: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_hybrid_score_concat: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_BEFORE)
    _rg = gen_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_GEN)
    compare(_rb, _rg, "hybrid_score_concat")
except Exception as _e:
    print(f"❌ L2 equivalence hybrid_score_concat: setup error — {type(_e).__name__}: {_e}")

# L3 — a single HybridScore-like object should concatenate to one row.
try:
    _rb = before_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_BEFORE[:1])
    _rg = gen_hybrid_score_concat(FIX_HYBRID_SCORE_CONCAT_HYBRID_SCORES_GEN[:1])
    assert _rg.height == 1
    compare(_rb, _rg, "L3 hybrid_score_concat single item")
except Exception as _e:
    print(f"❌ L3 hybrid_score_concat single item: {type(_e).__name__}: {_e}")

# L3 — empty input raises on both pandas.concat and pl.concat.
try:
    before_err = gen_err = None
    try:
        before_hybrid_score_concat([])
    except Exception as e:
        before_err = type(e)
    try:
        gen_hybrid_score_concat([])
    except Exception as e:
        gen_err = type(e)
    if before_err is not None and gen_err is not None and before_err not in (SyntaxError, NameError) and gen_err not in (SyntaxError, NameError):
        print(f"✅ L3 hybrid_score_concat empty list: both sides reject (before={before_err.__name__}, gen={gen_err.__name__})")
    else:
        print(f"❌ L3 hybrid_score_concat empty list: MISMATCH — before={before_err}, gen={gen_err}")
except Exception as _e:
    print(f"❌ L3 hybrid_score_concat empty list: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_hybrid_score_concat: OK, type= DataFrame
✅ L1 smoke before_hybrid_score_concat: OK
✅ L2 equivalence hybrid_score_concat: MATCH
✅ L3 edge hybrid_score_concat single item: MATCH
✅ L3 hybrid_score_concat empty list: both sides reject (before=ValueError, gen=ValueError)
